In [70]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, brier_score_loss, confusion_matrix
from sklearn.model_selection import train_test_split

training_dataset = pd.read_csv("encoded_training_table.csv")

## Chronological Splitting

In [83]:
target = training_dataset['responded']
general_model_dataset = training_dataset.copy()
general_model_dataset = general_model_dataset.drop(columns=['uuid', 'responded'])

split_index = int(len(training_dataset) * 0.80)

general_model_X_train = general_model_dataset.iloc[:split_index]
general_model_y_train = target.iloc[:split_index]
general_model_X_test = general_model_dataset.iloc[split_index:]
general_model_y_test = target.iloc[split_index:]

# general_model_X_train, general_model_X_test, general_model_y_train, general_model_y_test = train_test_split(
#     general_model_dataset, target, test_size=0.2, random_state=42, stratify=target
# )

## Data Scaling

In [84]:
scale_cols = [
    'battery_level',
    'light_level',
    'noise_level',
    'acc_avg',
    'notifications_30m',
    'app_events_30m',
    'unlocks_30m',
    'prev_notifications',
    'prev_response_rate',
    'recent_response_rate',
    'slot_prev_response_rate',
    'mins_since_prev_notification',
    'age'
]

scaler = StandardScaler()
general_model_X_train_scaled = general_model_X_train.copy()
general_model_X_test_scaled = general_model_X_test.copy()

general_model_X_train_scaled[scale_cols] = scaler.fit_transform(general_model_X_train[scale_cols])
general_model_X_test_scaled[scale_cols] = scaler.transform(general_model_X_test[scale_cols])

## Logistic Regression Model Training

In [85]:
general_model = LogisticRegression(max_iter=2000, random_state=42)
general_model.fit(general_model_X_train_scaled, general_model_y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [88]:
y_predictions = general_model.predict(general_model_X_test_scaled)
y_probability = general_model.predict_proba(general_model_X_test_scaled)[:, 1]

print("Accuracy:", round(accuracy_score(general_model_y_test, y_predictions), 2))
print("Precision:", round(precision_score(general_model_y_test, y_predictions, zero_division=0), 2))
print("Recall:", round(recall_score(general_model_y_test, y_predictions, zero_division=0), 2))
print("F1-score:", round(f1_score(general_model_y_test, y_predictions, zero_division=0), 2))
print("ROC-AUC:", round(roc_auc_score(general_model_y_test, y_probability), 2))
print("Log loss:", round(log_loss(general_model_y_test, y_probability), 2))
print("Brier score:", round(brier_score_loss(general_model_y_test, y_probability), 2))
print("Confusion Matrix:")
print(confusion_matrix(general_model_y_test, y_predictions))

Accuracy: 0.81
Precision: 0.54
Recall: 0.34
F1-score: 0.42
ROC-AUC: 0.78
Log loss: 0.43
Brier score: 0.14
Confusion Matrix:
[[16621  1339]
 [ 3058  1583]]


In [67]:
time_slots = [
    "early morning",
    "mid morning",
    "late morning",
    "afternoon",
    "evening",
    "night"
]

results = []

for slot in time_slots:

    # Copy the same user context.
    candidate = general_model_X_test_scaled.copy()

    # Reset all time-slot columns to 0.
    for name in time_slots[1:]:
        candidate[f"time_slot_{name}"] = 0

    # Activate the selected time slot.
    if(slot != "early morning"):
        candidate[f"time_slot_{slot}"] = 1

    # Predict the probability of responding.
    probabilities = general_model.predict_proba(candidate)[:, 1]

    results.append({
        "time_slot": slot,
        "average_response_probability": round(probabilities.mean() * 100, 2)
    })

# Display results from highest to lowest probability.
results = pd.DataFrame(results).sort_values(
    "average_response_probability",
    ascending=False
)

print(results.to_string(index=False))

print(
    "\nRecommended time slot:",
    results.iloc[0]["time_slot"]
)

    time_slot  average_response_probability
 late morning                         30.78
  mid morning                         28.80
      evening                         28.52
    afternoon                         28.26
        night                         20.34
early morning                         11.67

Recommended time slot: late morning


In [64]:
y_train = general_model_y_train
y_test = general_model_y_test

# Baseline 1: predict the majority class
majority_class = y_train.value_counts().idxmax()

majority_predictions = np.full(len(y_test), majority_class)

print("Baseline 1: majority class")
print("Majority class:", majority_class)

print("Accuracy:", accuracy_score(y_test, majority_predictions))


# Baseline 2: Predict the training response rate
training_response_rate = y_train.mean()

constant_probabilities = np.full(len(y_test), training_response_rate)

print("\nBaseline 2: historical response rate")
print("Training response rate:", round(training_response_rate, 4))
print("Brier score:", round(brier_score_loss(y_test, constant_probabilities), 4))
print("Log loss:", round(log_loss(y_test, constant_probabilities), 4))


# Evaluate your Logistic Regression model
y_predictions = general_model.predict(general_model_X_test_scaled)
y_probability = general_model.predict_proba(general_model_X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(general_model_y_test, y_predictions))
print("Precision:", precision_score(general_model_y_test, y_predictions, zero_division=0))
print("Recall:", recall_score(general_model_y_test, y_predictions, zero_division=0))
print("F1-score:", f1_score(general_model_y_test, y_predictions, zero_division=0))
print("ROC-AUC:", roc_auc_score(general_model_y_test, y_probability))
print("Log loss:", log_loss(general_model_y_test, y_probability))
print("Brier score:", brier_score_loss(general_model_y_test, y_probability))
print("Confusion Matrix:")
print(confusion_matrix(general_model_y_test, y_predictions))

Baseline 1: majority class
Majority class: 0
Accuracy: 0.794655103756471

Baseline 2: historical response rate
Training response rate: 0.2997
Brier score: 0.1721
Log loss: 0.5305
Accuracy: 0.8054510862351223
Precision: 0.5417522245037646
Recall: 0.34109028226675286
F1-score: 0.41861695094539203
ROC-AUC: 0.7839428661647974
Log loss: 0.42672627344042624
Brier score: 0.1372115250255239
Confusion Matrix:
[[16621  1339]
 [ 3058  1583]]


In [66]:

import pandas as pd

time_slots = [
    "early morning",
    "mid morning",
    "late morning",
    "afternoon",
    "evening",
    "night"
]

# Use the original dataset, before scaling.
train_original = training_dataset.iloc[:split_index].copy()
test_original = training_dataset.iloc[split_index:].copy()

# Recover each notification's actual time slot.
# Early morning is the omitted reference category.
def get_time_slot(row):
    for slot in time_slots[1:]:
        column = f"time_slot_{slot}"

        if row[column] == 1:
            return slot

    return "early morning"

train_original["time_slot"] = train_original.apply(
    get_time_slot, axis=1
)

test_original["time_slot"] = test_original.apply(
    get_time_slot, axis=1
)

# 1. Calculate response rates in the training data.
training_slot_rates = (
    train_original.groupby("time_slot")["responded"]
    .mean()
    .reindex(time_slots)
)

print("Training response rates:")
print((training_slot_rates * 100).round(2))

# 2. Choose the best slot using training data only.
best_baseline_slot = training_slot_rates.idxmax()

print("\nBest-time-slot baseline:", best_baseline_slot)

# 3. Evaluate notifications that actually occurred in that slot
# in the test data.
baseline_test_rows = test_original[
    test_original["time_slot"] == best_baseline_slot
]

if len(baseline_test_rows) > 0:
    baseline_test_rate = baseline_test_rows["responded"].mean()

    print("Test notifications in baseline slot:",
          len(baseline_test_rows))
    print("Observed test response rate:",
          round(baseline_test_rate * 100, 2), "%")
else:
    print("No test notifications were observed in this slot.")
print("original", round(train_original["responded"].mean() * 100, 2), "%")

Training response rates:
time_slot
early morning     3.92
mid morning      25.97
late morning     42.59
afternoon        42.62
evening          43.09
night            23.55
Name: responded, dtype: float64

Best-time-slot baseline: evening
Test notifications in baseline slot: 3051
Observed test response rate: 27.14 %
original 29.97 %
